Verfica a saúde do servidor (Funcionamento correto)

In [ ]:
import requests

r = requests.get('http://54.196.116.14:1026/version', auth=('user', 'pass'))
r.status_code
r.json()

Solicita série de dados temporais via método Get HTTP.

In [ ]:
import requests
import matplotlib.pyplot as plt
import numpy as np

# Função para obter os dados de luminosidade a partir da API
def obter_dados_potentiometer(lastN):
    url = f"http://54.196.116.14:8666/STH/v1/contextEntities/type/device/id/urn:ngsi-ld:device:001/attributes/p?lastN={lastN}"

    headers = {
        'fiware-service': 'smart',
        'fiware-servicepath': '/'
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        luminosity_data = data['contextResponses'][0]['contextElement']['attributes'][0]['values']
        return luminosity_data
    else:
        print(f"Erro ao obter dados: {response.status_code}")
        return []

# Função para criar e exibir o gráfico
def plotar_grafico(potentiometer_data):
    if not potentiometer_data:
        print("Nenhum dado disponível para plotar.")
        return

    potentiometer = [entry['attrValue'] for entry in potentiometer_data]
    tempos = [entry['recvTime'] for entry in potentiometer_data]

    plt.figure(figsize=(12, 6))
    plt.plot(tempos, potentiometer, marker='o', linestyle='-', color='r')
    plt.title('Gráfico de mediçoes do potênciometro em função do Tempo')
    plt.xlabel('Tempo')
    plt.ylabel('Valores')
    plt.xticks(rotation=90)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Solicitar ao usuário um valor "lastN" entre 1 e 100
while True:
    try:
        lastN = int(input("Digite um valor para lastN (entre 1 e 100): "))
        if 1 <= lastN <= 100:
            break
        else:
            print("O valor deve estar entre 1 e 100. Tente novamente.")
    except ValueError:
        print("Por favor, digite um número válido.")

# Obter os dados de luminosidade e plotar o gráfico
potentiometer_data = obter_dados_potentiometer(lastN)
plotar_grafico(potentiometer_data)


Análise dos pontos de minimo e máximo

In [ ]:
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 0) Extração dos dados
# -------------------------------
potentiometer = np.array([float(entry['attrValue']) for entry in potentiometer_data])
tempos = np.array([entry['recvTime'] for entry in potentiometer_data])

# Caso recvTime sejam strings, criamos um eixo numérico
x = np.arange(len(potentiometer))

# -------------------------------
# 1) Máximos e mínimos locais
# -------------------------------
max_idx, _ = find_peaks(potentiometer)
min_idx, _ = find_peaks(-potentiometer)

# -------------------------------
# 2) Máximos e mínimos globais
# -------------------------------
idx_max_global = np.argmax(potentiometer)
idx_min_global = np.argmin(potentiometer)

y_max = potentiometer[idx_max_global]
y_min = potentiometer[idx_min_global]

# -------------------------------
# 3) GRÁFICO
# -------------------------------
plt.figure(figsize=(12,6))

plt.plot(x, potentiometer, label="Sinal do Potenciômetro")
plt.plot(x[max_idx], potentiometer[max_idx], 'ro', label='Máximos locais')
plt.plot(x[min_idx], potentiometer[min_idx], 'go', label='Mínimos locais')

# -------------------------------
# 4) Anotações com valores numéricos
# -------------------------------

plt.annotate(f"Máximo global\nValor = {y_max:.2f}",
             xy=(idx_max_global, y_max),
             xytext=(idx_max_global + 15, y_max + 0.15*y_max),
             arrowprops=dict(arrowstyle="->"),
             fontsize=10,
             bbox=dict(boxstyle="round", fc="white", ec="black", alpha=0.8))

plt.annotate(f"Mínimo global\nValor = {y_min:.2f}",
             xy=(idx_min_global, y_min),
             xytext=(idx_min_global + 15, y_min - 0.15*abs(y_min)),
             arrowprops=dict(arrowstyle="->"),
             fontsize=10,
             bbox=dict(boxstyle="round", fc="white", ec="black", alpha=0.8))

plt.title("Análise do Sinal do Potenciômetro")
plt.xlabel("Índice / Tempo")
plt.ylabel("Valor lido")
plt.grid(True)
plt.legend()
plt.show()



Análise da taxa de variação

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# -------------------------------
# 0) Extração dos dados
# -------------------------------
potentiometer = np.array([float(entry['attrValue']) for entry in potentiometer_data])
tempos = np.array([entry['recvTime'] for entry in potentiometer_data])

# Criar eixo numérico caso recvTime sejam strings
x = np.arange(len(potentiometer))

# -------------------------------
# 1) Taxa de variação (gradiente)
# -------------------------------
gradiente = np.gradient(potentiometer)

# -------------------------------
# 2) Gráfico do gradiente
# -------------------------------
plt.figure(figsize=(12,4))
plt.plot(x, gradiente, color="orange", label="Taxa de variação (gradiente)")

plt.axhline(0, color="gray", linestyle="--")  # linha de referência
plt.title("Taxa de Variação do Sinal (Gradiente)")
plt.xlabel("Índice / Tempo")
plt.ylabel("ΔValor")
plt.grid(True)
plt.legend()

plt.show()


Análise Estatistica (Histograma)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 0) Extração dos dados
# -------------------------------
potentiometer = np.array([float(entry['attrValue']) for entry in potentiometer_data])

# -------------------------------
# 1) Estatísticas básicas
# -------------------------------
media = np.mean(potentiometer)
desvio = np.std(potentiometer)

print(f"Valor médio do sinal: {media:.4f}")
print(f"Desvio padrão: {desvio:.4f}")

# -------------------------------
# 2) Histograma
# -------------------------------
plt.figure(figsize=(12,5))

plt.hist(potentiometer, bins=30, color="skyblue", edgecolor="black", alpha=0.7, density=True)
plt.axvline(media, color="red", linestyle="--", linewidth=2, label=f"Média = {media:.2f}")

# Comentário anotado no gráfico
plt.annotate(f"Média = {media:.2f}",
             xy=(media, 0),
             xytext=(media + (potentiometer.max()-potentiometer.min())*0.05,
                     plt.ylim()[1]*0.8),
             arrowprops=dict(arrowstyle="->"),
             fontsize=11,
             bbox=dict(boxstyle="round", fc="white", ec="black", alpha=0.8))

plt.title("Histograma do Sinal do Potenciômetro")
plt.xlabel("Valor medido")
plt.ylabel("Densidade de Probabilidade")
plt.grid(True, alpha=0.4)
plt.legend()
plt.show()


Análise Avançada

In [ ]:
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 0) Extração dos dados
# -------------------------------
potentiometer = np.array([float(entry['attrValue']) for entry in potentiometer_data])
tempos = np.array([entry['recvTime'] for entry in potentiometer_data])

x = np.arange(len(potentiometer))

# -------------------------------
# 1) Estatísticas
# -------------------------------
media = np.mean(potentiometer)

# -------------------------------
# 2) Máximos e mínimos locais
# -------------------------------
max_idx, _ = find_peaks(potentiometer)
min_idx, _ = find_peaks(-potentiometer)

# Máximos e mínimos globais
idx_max_global = np.argmax(potentiometer)
idx_min_global = np.argmin(potentiometer)

y_max = potentiometer[idx_max_global]
y_min = potentiometer[idx_min_global]

# -------------------------------
# 3) Gradiente
# -------------------------------
gradiente = np.gradient(potentiometer)

# -------------------------------
# 4) GRÁFICO ÚNICO COMPLETO
# -------------------------------
plt.figure(figsize=(15,7))

# Sinal original
plt.plot(x, potentiometer, label="Sinal do Potenciômetro", linewidth=2)

# Máximos e mínimos locais
plt.plot(x[max_idx], potentiometer[max_idx], 'ro', label='Máximos locais')
plt.plot(x[min_idx], potentiometer[min_idx], 'go', label='Mínimos locais')

# Gradiente
plt.plot(x, gradiente, color="orange", alpha=0.7, label="Taxa de Variação (Gradiente)")

# Linha da média
plt.axhline(media, color="purple", linestyle="--", linewidth=2, label=f"Média = {media:.2f}")

# -------------------------------
# 5) Anotações COM valores
# -------------------------------

plt.annotate(
    f"Máximo global\n{y_max:.2f}",
    xy=(idx_max_global, y_max),
    xytext=(idx_max_global + 20, y_max + 0.2 * y_max),
    arrowprops=dict(arrowstyle="->"),
    fontsize=10,
    bbox=dict(boxstyle="round", fc="white", ec="black", alpha=0.85)
)

plt.annotate(
    f"Mínimo global\n{y_min:.2f}",
    xy=(idx_min_global, y_min),
    xytext=(idx_min_global + 20, y_min - 0.2 * abs(y_min)),
    arrowprops=dict(arrowstyle="->"),
    fontsize=10,
    bbox=dict(boxstyle="round", fc="white", ec="black", alpha=0.85)
)

# Anotação da média
plt.annotate(
    f"Média = {media:.2f}",
    xy=(5, media),
    xytext=(50, media + (potentiometer.max() - potentiometer.min()) * 0.1),
    arrowprops=dict(arrowstyle="->"),
    fontsize=11,
    bbox=dict(boxstyle="round", fc="white", ec="purple", alpha=0.8)
)

# -------------------------------
# Rótulos
# -------------------------------
plt.title("Análise Completa do Sinal do Potenciômetro")
plt.xlabel("Índice / Tempo")
plt.ylabel("Valor")
plt.grid(True)
plt.legend()
plt.show()
